# Import important Libaries

In [21]:
# cd Desktop/Nottingham/3rd/Dissertation/Code/Comp3003
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from tensorflow import keras
from keras.layers import Dense
from keras.callbacks import EarlyStopping, Callback, ModelCheckpoint, ReduceLROnPlateau

In [2]:
df = pd.read_csv('selected_features.csv')
df.head(10)

,Dst Port,Flow Duration,Flow Pkts/s,Fwd Header Len,Bwd Header Len,Fwd Pkts/s,Bwd Pkts/s,Subflow Fwd Byts,Init Fwd Win Byts,Init Bwd Win Byts,Fwd Act Data Pkts,Label
0,443,116087899,3.445665e-01,432,524,0.180897,0.163669,750,8192,123,15,0
1,21,1,2.000000e+06,40,20,1000000.000000,1000000.000000,0,26883,0,0,1
2,3389,1206036,1.243744e+01,172,152,6.633301,5.804139,1148,8192,62852,5,0
3,53,1273,1.571092e+03,8,8,785.545954,785.545954,32,-1,-1,0,0
4,21,2,1.000000e+06,40,20,500000.000000,500000.000000,0,26883,0,0,1
5,21,1,2.000000e+06,40,20,1000000.000000,1000000.000000,0,26883,0,0,1
6,443,131855,1.137613e+02,172,148,60.672709,53.088620,1087,8192,5414,4,0
7,80,77787270,2.828226e-01,252,296,0.154267,0.128556,455,8192,946,8,0
8,21,2,1.000000e+06,40,20,500000.000000,500000.000000,0,26883,0,0,1
9,53,25471,7.852067e+01,8,8,39.260335,39.260335,49,-1,-1,0,0


---
Let's take a look at how much data we have to work with. 

In [3]:
print("This Dataset has {} rows and {} columns".format(df.shape[0], df.shape[1]))

This Dataset has 49434 rows and 12 columns


---
Checking that all important colums are present. 

In [4]:
df.columns

Index(['Dst Port', 'Flow Duration', 'Flow Pkts/s', 'Fwd Header Len',
       'Bwd Header Len', 'Fwd Pkts/s', 'Bwd Pkts/s', 'Subflow Fwd Byts',
       'Init Fwd Win Byts', 'Init Bwd Win Byts', 'Fwd Act Data Pkts', 'Label'],
      dtype='object')

---
Listing the numbers of unique variables per feature. 

In [5]:
print(df.nunique())

Dst Port              3088
Flow Duration        23317
Flow Pkts/s          24249
Fwd Header Len         290
Bwd Header Len         407
Fwd Pkts/s           24131
Bwd Pkts/s           21788
Subflow Fwd Byts      1574
Init Fwd Win Byts      617
Init Bwd Win Byts      707
Fwd Act Data Pkts       52
Label                    3
dtype: int64


---
Ensuring all negative values are removed. 

In [6]:
df_cleaned = df[(df >= 0).all(axis=1)]

### Spliting target variable from features 

In [7]:
#separating input and output variables
x = df_cleaned.drop(['Label'], axis=1)
y = df_cleaned['Label']
print(x)

       Dst Port  Flow Duration   Flow Pkts/s  Fwd Header Len  Bwd Header Len  \
0           443      116087899  3.445665e-01             432             524   
1            21              1  2.000000e+06              40              20   
2          3389        1206036  1.243744e+01             172             152   
4            21              2  1.000000e+06              40              20   
5            21              1  2.000000e+06              40              20   
...         ...            ...           ...             ...             ...   
49429        80          74113  9.445037e+01              92              72   
49430       443        7216573  2.217119e+00             152             192   
49431        22         388232  1.133343e+02             776             648   
49432      3389        5328190  3.190577e+00             212             152   
49433        21              1  2.000000e+06              40              20   

           Fwd Pkts/s      Bwd Pkts/s  

### Split DS
70% training 30% rest
15% valid 15% test

In [19]:
# Split data into training (70%) and temporary set (30%)
X_train, X_temp, y_train, y_temp = train_test_split(x, y, test_size=0.3, random_state=21)

# Split the temporary set equally into validation (15%) and testing (15%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=21)

print(f"Training set: {len(X_train)} rows")
print(f"Validation set: {len(X_val)} rows")
print(f"Testing set: {len(X_test)} rows")

Training set: 23811 rows
Validation set: 5103 rows
Testing set: 5103 rows


Setting up a c

In [9]:
Classifier_accuracy = []

### Defining the Deep Neural Network

In [16]:
# Define and compile model
model = keras.Sequential()
model.add(Dense(28 , input_shape=(10,) , activation="relu" , name="Hidden_Layer_1"))
model.add(Dense(10 , activation="relu" , name="Hidden_Layer_2"))
model.add(Dense(3, activation="softmax", name="Output_Layer")) 
opt = keras.optimizers.Adam(learning_rate=0.01)
model.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=['accuracy'])
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 Hidden_Layer_1 (Dense)      (None, 28)                308       
                                                                 
 Hidden_Layer_2 (Dense)      (None, 10)                290       
                                                                 
 Output_Layer (Dense)        (None, 3)                 33        
                                                                 
Total params: 631
Trainable params: 631
Non-trainable params: 0
_________________________________________________________________


---
## Model Buidling

Creating an initial model and compare it's performance. 

In [20]:
# fit model
history_org = model.fit(
    X_train, 
    y_train, 
    batch_size=32, 
    epochs=100, 
    # 要跑多少遍你就调epochs
    verbose=2,
    callbacks=[early_stopping, checkpoint, reduce_lr], 
    validation_data=(X_test,y_test), 
    shuffle=True, 
    class_weight=None, 
    sample_weight=None, 
    initial_epoch=0)

NameError: name 'early_stopping' is not defined

### Plotting Loss v/s Epochs

In [ ]:
loss = history_org.history['loss']
val_loss = history_org.history['val_loss']
epochs = range(1, len(loss) + 1)
plt.plot(epochs, loss, 'g', label = 'Training Loss')
plt.plot(epochs, val_loss, 'r', label = 'Validation Loss')
plt.title('Loss v/s No. of epochs')
plt.xlabel('Number of Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

### Plotting Accuracy v/s Epochs

In [ ]:
loss = history_org.history['accuracy']
val_loss = history_org.history['val_accuracy']
plt.plot(epochs, loss, 'g', label = 'Training accuracy')
plt.plot(epochs, val_loss, 'r', label = 'Validation accuracy')
plt.title('Accuracy Scores v/s Number of Epochs')
plt.xlabel('Number of Epochs')
plt.ylabel('Accuracy Score')
plt.legend()
plt.show()